# MultiCycPermea — Per-setting hard sample deep dive

For each split (ID / OD / Cliff_ratio; OD_Murcko added if its dump is present), this notebook produces:

1. **Top-K hardest molecules** with SMILES, structure image, full descriptor row.
2. **Hardness driver** — fit a `GradientBoostingRegressor` to (descriptors → |residual|) and report the top features (permutation importance).
3. **2D hard regions** — descriptor pair scatter with hard samples highlighted.
4. **Sequence-level** — monomer-length distribution of hard vs rest.
5. **Cross-split overlap** — how many hard samples are shared across splits.
6. **Per-setting written summary** at the end.

In [ ]:
from pathlib import Path
import math, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, spearmanr
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, DataStructs
from IPython.display import display

ROOT  = Path('/ssd0/sohyun/cyclic_peptide/cyclic_peptide_permeability')
MCP   = ROOT / 'MultiCycPermea' / 'DL'
PRED  = MCP / 'pred_data'
DATA  = MCP / 'data' / 'ours'

DESC_COLS = [
    'Sequence_LogP','Sequence_TPSA','Monomer_Length','Monomer_Length_in_Main_Chain',
    'MolWt','HeavyAtomMolWt','ExactMolWt','NumValenceElectrons',
    'qed','MaxPartialCharge','MinPartialCharge',
    'FpDensityMorgan1','FpDensityMorgan2','FpDensityMorgan3',
    'NumHDonors','NumHAcceptors','NumRotatableBonds','NumAromaticRings','RingCount',
    'TPSA','MolLogP','LabuteASA','EPSA','PSA','_3DPSA',
]

def latest(prefix):
    hits = sorted(PRED.glob(f'{prefix}*/best_test_predictions.csv'))
    return hits[-1] if hits else None

DUMPS = {}
for s, prefix in [
    ('ID','ours_ID_dump_'),
    ('OD','ours_OD_dump_'),
    ('Cliff_ratio','ours_Cliff_ratio_dump_'),
    ('OD_Murcko','ours_OD_Murcko_dump_'),
]:
    p = latest(prefix)
    if p is not None:
        DUMPS[s] = p
print('found dumps:'); [print(f'  {k}: {v.relative_to(ROOT)}') for k, v in DUMPS.items()]

In [ ]:
def load_split(split, pred_path):
    pred = pd.read_csv(pred_path)
    test = pd.read_csv(DATA / f'{split}_test.csv')
    keep = ['CycPeptMPDB_ID','PAMPA','SMILES','Sequence'] + [c for c in DESC_COLS if c in test.columns]
    df = pred.merge(test[keep], on='CycPeptMPDB_ID', how='left')
    df['residual'] = df['y_pred'] - df['y_true']
    df['abs_residual'] = df['residual'].abs()
    df['split'] = split
    return df

frames = {s: load_split(s, p) for s, p in DUMPS.items()}
summary_rows = []
for s, df in frames.items():
    summary_rows.append({
        'split': s, 'N': len(df),
        'MAE': df['abs_residual'].mean(),
        'RMSE': (df['residual']**2).mean()**0.5,
        'bias': df['residual'].mean(),
        'p90_|res|': df['abs_residual'].quantile(0.9),
    })
summary = pd.DataFrame(summary_rows).round(4)
summary

## 1. Top-20 hardest molecules per split (SMILES + structures)

In [ ]:
def show_hardest(df, k=20):
    cols = ['CycPeptMPDB_ID','y_true','y_pred','residual','abs_residual',
            'MolLogP','TPSA','NumHDonors','NumHAcceptors','MolWt','Monomer_Length','qed','SMILES']
    cols = [c for c in cols if c in df.columns]
    return df.nlargest(k, 'abs_residual')[cols]

hardest_per_split = {s: show_hardest(df, k=20) for s, df in frames.items()}
for s, h in hardest_per_split.items():
    print(f'\n=== {s}: top-20 hardest ===')
    display(h.style.format({'y_true':'{:.3f}','y_pred':'{:.3f}','residual':'{:+.3f}','abs_residual':'{:.3f}',
                            'MolLogP':'{:.2f}','TPSA':'{:.1f}','MolWt':'{:.1f}','qed':'{:.3f}'}))

In [ ]:
# 2D-structure grids for top-8 hardest per split
def draw_top(df, k=8, title=''):
    sub = df.nlargest(k, 'abs_residual').reset_index(drop=True)
    mols = [Chem.MolFromSmiles(s) for s in sub['SMILES']]
    valid = [(m, i) for i, m in enumerate(mols) if m is not None]
    mols_ok = [m for m, _ in valid]
    legends = [
        f"id={sub['CycPeptMPDB_ID'].iloc[i]} y={sub['y_true'].iloc[i]:.2f} \u0177={sub['y_pred'].iloc[i]:.2f} \u0394={sub['residual'].iloc[i]:+.2f}"
        for _, i in valid
    ]
    img = Draw.MolsToGridImage(mols_ok, molsPerRow=4, subImgSize=(260, 220), legends=legends)
    print(f'\n=== {title} ===')
    display(img)

for s, df in frames.items():
    draw_top(df, k=8, title=f'{s} — top-8 hardest')

## 2. Hardness driver via Gradient Boosting (permutation importance)

Fit GBR: descriptors → |residual|. Report the descriptors whose permutation most degrades the model — those are the strongest hardness predictors **within that split**.

In [ ]:
def hardness_driver(df, n_features=15, random_state=0):
    feats = [c for c in DESC_COLS if c in df.columns]
    X = df[feats].apply(pd.to_numeric, errors='coerce')
    # drop columns that are all-NaN or mostly NaN (>50%)
    keep = X.columns[X.isna().mean() < 0.5].tolist()
    X = X[keep]
    # impute remaining NaN with median
    X = X.fillna(X.median(numeric_only=True))
    y = df['abs_residual'].values
    if X.shape[0] == 0 or X.shape[1] == 0:
        return None, float('nan'), pd.DataFrame(columns=['feature','perm_importance_mean','perm_importance_std'])
    gbr = GradientBoostingRegressor(max_depth=3, n_estimators=300, learning_rate=0.05, random_state=random_state)
    gbr.fit(X.values, y)
    score = gbr.score(X.values, y)
    perm = permutation_importance(gbr, X.values, y, n_repeats=10, random_state=random_state, n_jobs=4)
    order = np.argsort(perm.importances_mean)[::-1][:n_features]
    imp = pd.DataFrame({
        'feature': [keep[i] for i in order],
        'perm_importance_mean': perm.importances_mean[order],
        'perm_importance_std':  perm.importances_std[order],
    })
    return gbr, score, imp

driver_results = {}
for s, df in frames.items():
    gbr, r2, imp = hardness_driver(df)
    driver_results[s] = (gbr, r2, imp)
    print(f'\n=== {s}: GBR (descriptors -> |residual|) R^2_train={r2:.3f}; top features by permutation importance ===')
    display(imp.head(10).style.format({'perm_importance_mean':'{:.4f}','perm_importance_std':'{:.4f}'}))

In [ ]:
# Heatmap of permutation importance across splits
imp_table = {}
for s, (gbr, r2, imp) in driver_results.items():
    imp_table[s] = dict(zip(imp['feature'], imp['perm_importance_mean']))
imp_df = pd.DataFrame(imp_table).fillna(0)
# select top features by max across splits
imp_df = imp_df.reindex(imp_df.max(axis=1).sort_values(ascending=False).head(15).index)
plt.figure(figsize=(7, 0.4*len(imp_df)+1.5))
sns.heatmap(imp_df, cmap='viridis', annot=True, fmt='.3f', cbar_kws={'label':'permutation importance (Δ R^2)'} )
plt.title('Hardness drivers across splits')
plt.tight_layout(); plt.show()

## 3. 2D hard region maps
Plot the test set on a key descriptor pair and highlight the hardest 10 %.

In [ ]:
PAIRS = [('MolLogP','TPSA'), ('MolLogP','NumHDonors'), ('TPSA','MolWt')]
fig, axes = plt.subplots(len(PAIRS), len(frames), figsize=(4.5*len(frames), 4*len(PAIRS)), squeeze=False)
for j, (s, df) in enumerate(frames.items()):
    thr = df['abs_residual'].quantile(0.9)
    easy = df[df['abs_residual'] <  thr]
    hard = df[df['abs_residual'] >= thr]
    for i, (x, y) in enumerate(PAIRS):
        ax = axes[i, j]
        if x not in df.columns or y not in df.columns:
            ax.set_visible(False); continue
        ax.scatter(easy[x], easy[y], s=10, c='lightgray', alpha=0.6, label='rest')
        sc = ax.scatter(hard[x], hard[y], s=28, c=hard['residual'], cmap='RdBu_r', vmin=-1.5, vmax=1.5,
                        edgecolor='k', linewidth=0.3, label='hardest 10%')
        ax.set_xlabel(x); ax.set_ylabel(y)
        if i == 0: ax.set_title(s)
        if j == len(frames)-1:
            plt.colorbar(sc, ax=ax, label='residual')
plt.tight_layout(); plt.show()

## 4. Sequence-level (monomer length)

In [ ]:
fig, axes = plt.subplots(1, len(frames), figsize=(4.5*len(frames), 3.8), sharey=True)
if len(frames) == 1: axes = [axes]
for ax, (s, df) in zip(axes, frames.items()):
    if 'Monomer_Length' not in df.columns:
        ax.set_visible(False); continue
    thr = df['abs_residual'].quantile(0.9)
    counts_all  = df['Monomer_Length'].value_counts().sort_index()
    counts_hard = df.loc[df['abs_residual']>=thr,'Monomer_Length'].value_counts().reindex(counts_all.index, fill_value=0)
    rate = counts_hard / counts_all
    ax2 = ax.twinx()
    ax.bar(counts_all.index.astype(str), counts_all.values, color='steelblue', alpha=0.6, label='all test')
    ax.bar(counts_hard.index.astype(str), counts_hard.values, color='crimson', alpha=0.85, label='hard 10%')
    ax2.plot(counts_all.index.astype(str), rate.values, 'k-o', linewidth=1.5, label='hard rate')
    ax2.set_ylabel('hard rate', color='k')
    ax.set_title(s); ax.set_xlabel('Monomer length')
    if ax is axes[0]:
        ax.set_ylabel('count'); ax.legend(loc='upper left')
        ax2.legend(loc='upper right')
plt.tight_layout(); plt.show()

## 5. Cross-split overlap of hard samples
How many hard-10% molecules are shared between splits? Only IDs present in both test sets are considered.

In [ ]:
hard_sets = {}
for s, df in frames.items():
    thr = df['abs_residual'].quantile(0.9)
    hard_sets[s] = set(df.loc[df['abs_residual']>=thr,'CycPeptMPDB_ID'])

# overlap matrix
splits = list(hard_sets)
M = pd.DataFrame(index=splits, columns=splits, dtype=int)
for a in splits:
    for b in splits:
        M.loc[a,b] = len(hard_sets[a] & hard_sets[b])
print('|hard ∩| matrix:')
display(M)

# also intersect the test ID *populations* (not just hard) so we know the denominator
all_sets = {s: set(df['CycPeptMPDB_ID']) for s, df in frames.items()}
print('\nshared test population (pairs):')
for i,a in enumerate(splits):
    for b in splits[i+1:]:
        common = all_sets[a] & all_sets[b]
        ha = hard_sets[a] & common
        hb = hard_sets[b] & common
        both = ha & hb
        print(f'  {a}↔{b}: shared test={len(common)}, hard in {a}={len(ha)}, hard in {b}={len(hb)}, hard in both={len(both)}')

## 6. Per-setting written summary

In [ ]:
def summarise(s, df, imp):
    thr = df['abs_residual'].quantile(0.9)
    hard = df[df['abs_residual']>=thr]
    bias = df['residual'].mean()
    mae  = df['abs_residual'].mean()
    # quartile bias
    q = pd.qcut(df['y_true'], 4, labels=['Q1','Q2','Q3','Q4'])
    bias_q = df.groupby(q, observed=False)['residual'].mean()
    top_feats = imp['feature'].head(5).tolist()
    # within-hard mean of top features
    deltas = []
    for f in top_feats:
        if f not in df.columns: continue
        a = pd.to_numeric(hard[f], errors='coerce')
        b = pd.to_numeric(df.loc[df['abs_residual']<thr, f], errors='coerce')
        deltas.append((f, a.mean(), b.mean(), a.mean()-b.mean()))
    direction = ('over-prediction (model says more permeable)' if bias>0 else 'under-prediction (model says less permeable)') if abs(bias)>0.1 else 'no strong global bias'
    text = []
    text.append(f'### {s}')
    text.append(f'- N={len(df)}, MAE={mae:.3f}, bias={bias:+.3f} → {direction}.')
    text.append(f'- |residual| by y_true quartile: ' + ', '.join(f'{q}={v:.2f}' for q, v in df.groupby(q, observed=False)["abs_residual"].mean().items()))
    text.append(f'- signed residual by quartile: ' + ', '.join(f'{q}={v:+.2f}' for q,v in bias_q.items()))
    text.append(f'- top hardness drivers (permutation): {", ".join(top_feats)}')
    if deltas:
        text.append(f'- hard-10% means (vs rest):')
        for f, a, b, d in deltas:
            text.append(f'  • {f}: {a:.3f} (hard) vs {b:.3f} (rest), Δ={d:+.3f}')
    return '\n'.join(text)

from IPython.display import Markdown
for s, df in frames.items():
    _, _, imp = driver_results[s]
    display(Markdown(summarise(s, df, imp)))

In [ ]:
# Save artefacts: union hard sample CSV with all descriptors + per-split summary text
OUT_CSV  = ROOT / 'eda' / 'hard_samples_deep_all.csv'
OUT_TXT  = ROOT / 'eda' / 'hard_samples_summary.md'
parts = []
for s, df in frames.items():
    thr = df['abs_residual'].quantile(0.9)
    h = df[df['abs_residual']>=thr].copy()
    h['split'] = s
    parts.append(h)
pd.concat(parts, ignore_index=True).to_csv(OUT_CSV, index=False)
with open(OUT_TXT, 'w') as f:
    f.write('# Per-setting hard sample summary\n\n')
    for s, df in frames.items():
        _, _, imp = driver_results[s]
        f.write(summarise(s, df, imp))
        f.write('\n\n')
print(f'wrote {OUT_CSV} and {OUT_TXT}')